<a href="https://colab.research.google.com/github/leroymakitaloubaki-web/license-plate-recognition-yolov8/blob/main/parking_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELLULE 1 — Setup & téléchargement PKLot
# ============================================================

# --- 1. Dépendances ---
!pip install torch torchvision matplotlib seaborn kaggle -q

# --- 2. Monte Google Drive (pour sauvegarder le modèle) ---
from google.colab import drive
drive.mount('/content/drive')

# --- 3. Vérifie le GPU ---
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device : {device}")
if device == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

# --- 4. Upload ta clé Kaggle (kaggle.json) ---
from google.colab import files
print("\n📁 Upload ton fichier kaggle.json :")
uploaded = files.upload()

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Clé Kaggle configurée")

# --- 5. Téléchargement PKLot ---
print("\n⬇️ Téléchargement PKLot...")
!kaggle datasets download -d ammarnassanalhajali/pklot-dataset -p /content/pklot --unzip -q
print("✅ Dataset téléchargé")

# --- 6. Exploration de la structure ---
import os
for root, dirs, files_list in os.walk('/content/pklot'):
    level = root.replace('/content/pklot', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:  # N'affiche pas trop de profondeur
        dirs.clear()

KeyboardInterrupt: 

In [ ]:
# Cellule d'exploration — structure complète PKLot
import os

for root, dirs, files_list in os.walk('/content/pklot'):
    level = root.replace('/content/pklot', '').count(os.sep)
    if level <= 3:  # On descend plus profond
        indent = '  ' * level
        nb_files = len(files_list)
        print(f"{indent}{os.path.basename(root)}/ {'→ ' + str(nb_files) + ' fichiers' if nb_files > 0 else ''}")

In [ ]:
# Vérification des classes
import os

for split in ['train', 'valid', 'test']:
    path = f'/content/pklot/{split}'
    classes = os.listdir(path)
    print(f"{split}/ → classes : {classes}")


In [ ]:
import os
print(os.listdir('/content/pklot/train'))

In [ ]:
# Exploration du fichier d'annotations COCO
import json, os

with open('/content/pklot/train/_annotations.coco.json', 'r') as f:
    data = json.load(f)

print("Clés principales:", list(data.keys()))
print(f"\nNombre d'images : {len(data['images'])}")
print(f"Nombre d'annotations : {len(data['annotations'])}")
print(f"Catégories : {data['categories']}")
print("\n--- Exemple d'image ---")
print(data['images'][0])
print("\n--- Exemple d'annotation ---")
print(data['annotations'][0])

In [ ]:
# ============================================================
# CELLULE 2 — Extraction des crops PKLot + Dataset PyTorch
# ============================================================

import json, os, cv2
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# --- 1. Paramètres ---
DATA_ROOT   = '/content/pklot'
CROP_ROOT   = '/content/pklot_crops'
SPLITS      = ['train', 'valid', 'test']
CLASS_NAMES = ['space-empty', 'space-occupied']   # id 1 → 0, id 2 → 1
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2

# --- 2. Extraction des crops depuis les bounding boxes ---
def extract_crops(split):
    ann_path = os.path.join(DATA_ROOT, split, '_annotations.coco.json')
    img_dir  = os.path.join(DATA_ROOT, split)
    out_dirs = {
        1: os.path.join(CROP_ROOT, split, 'space-empty'),
        2: os.path.join(CROP_ROOT, split, 'space-occupied')
    }
    for d in out_dirs.values():
        os.makedirs(d, exist_ok=True)

    with open(ann_path, 'r') as f:
        coco = json.load(f)

    # Dictionnaire id_image → nom de fichier
    id2file = {img['id']: img['file_name'] for img in coco['images']}

    saved, skipped = 0, 0
    for ann in coco['annotations']:
        cat_id = ann['category_id']
        if cat_id not in (1, 2):      # ignorer la catégorie parente 'spaces'
            skipped += 1
            continue

        img_path = os.path.join(img_dir, id2file[ann['image_id']])
        x, y, w, h = [int(v) for v in ann['bbox']]

        # Sécurité : bbox valide
        if w < 5 or h < 5:
            skipped += 1
            continue

        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue

        H, W = img.shape[:2]
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(W, x + w), min(H, y + h)

        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            skipped += 1
            continue

        out_name = f"{ann['id']}.jpg"
        out_path = os.path.join(out_dirs[cat_id], out_name)
        cv2.imwrite(out_path, crop)
        saved += 1

    print(f"  [{split}] Crops sauvegardés : {saved} | Ignorés : {skipped}")

print("⏳ Extraction des crops (peut prendre 2-3 min)...")
for split in SPLITS:
    extract_crops(split)
print("✅ Extraction terminée")

# --- 3. Vérification de la répartition ---
print("\n📊 Répartition des crops :")
for split in SPLITS:
    for cls in CLASS_NAMES:
        path = os.path.join(CROP_ROOT, split, cls)
        n = len(os.listdir(path))
        print(f"  {split}/{cls} : {n}")

# --- 4. Transforms ---
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- 5. Dataset PyTorch ---
class ParkingCropDataset(Dataset):
    def __init__(self, split, transform=None):
        self.samples   = []
        self.transform = transform
        for label, cls in enumerate(CLASS_NAMES):       # 0=empty, 1=occupied
            folder = os.path.join(CROP_ROOT, split, cls)
            for fname in os.listdir(folder):
                if fname.endswith('.jpg'):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# --- 6. DataLoaders ---
train_dataset = ParkingCropDataset('train', train_transform)
valid_dataset = ParkingCropDataset('valid', val_transform)
test_dataset  = ParkingCropDataset('test',  val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f"\n✅ DataLoaders prêts :")
print(f"  Train  : {len(train_dataset)} crops | {len(train_loader)} batches")
print(f"  Valid  : {len(valid_dataset)} crops | {len(valid_loader)} batches")
print(f"  Test   : {len(test_dataset)}  crops | {len(test_loader)} batches")

# --- 7. Visualisation de quelques crops ---
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle("Aperçu des crops PKLot", fontsize=13)

for label, cls in enumerate(CLASS_NAMES):
    folder = os.path.join(CROP_ROOT, 'train', cls)
    files  = os.listdir(folder)[:6]
    for j, fname in enumerate(files):
        img = Image.open(os.path.join(folder, fname)).convert('RGB')
        axes[label][j].imshow(img)
        axes[label][j].axis('off')
        if j == 0:
            axes[label][j].set_ylabel(cls, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELLULE 3 — Fine-tuning ResNet-50
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.optim.lr_scheduler import StepLR
import time

# --- 1. Chargement ResNet-50 pré-entraîné ImageNet ---
model = models.resnet50(weights='IMAGENET1K_V1')

# --- 2. Geler toutes les couches sauf la dernière ---
for param in model.parameters():
    param.requires_grad = False

# --- 3. Remplacer le classifieur final (2 classes) ---
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)
print(f"✅ ResNet-50 chargé sur {device}")
print(f"   Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# --- 4. Loss + Optimiseur + Scheduler ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=3, gamma=0.1)

# --- 5. Boucle d'entraînement ---
EPOCHS     = 5
best_acc   = 0.0
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

print(f"\n🚀 Entraînement sur {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    t0 = time.time()

    # --- TRAIN ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct      += predicted.eq(labels).sum().item()
        total        += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- VALIDATION ---
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct  / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # Sauvegarde du meilleur modèle
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_pklot_resnet50.pth')
        saved = "💾 sauvegardé"
    else:
        saved = ""

    elapsed = time.time() - t0
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
          f"{elapsed:.0f}s {saved}")

    scheduler.step()

print(f"\n✅ Entraînement terminé — Meilleure Val Acc : {best_acc:.4f}")

# --- 6. Courbes d'apprentissage ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses,   label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(train_accs, label='Train')
axes[1].plot(val_accs,   label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 46.9MB/s]


✅ ResNet-50 chargé sur cpu
   Paramètres entraînables : 4,098

🚀 Entraînement sur 5 epochs...

